In [1]:
import os
import json
import math
import random
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import GPT2Tokenizer

torch.manual_seed(42)
random.seed(42)

BASE_DIR = r"C:\Users\MIND & MATTER\Desktop\ScratchLLM-95_conversational"

CONFIG = {
    "vocab_size": 50257,
    "max_seq_len": 512,
    "embed_dim": 512,
    "num_heads": 8,
    "num_layers": 12,
    "d_ff": 2730,
    "dropout": 0.1,

    "lora_r": 32,
    "lora_alpha": 64,
    "lora_dropout": 0.05,
    "target_modules": ["qkv_proj", "out_proj"],

    "batch_size": 8,
    "grad_accum_steps": 4,
    "lr": 2e-4,
    "warmup_steps": 200,
    "epochs": 5,
    "grad_clip": 0.8,
    "eval_every_steps": 200,
    "save_every_steps": 200,

    "device": "cuda" if torch.cuda.is_available() else "cpu",
}

print("Device:", CONFIG["device"])
print(CONFIG)

Device: cuda
{'vocab_size': 50257, 'max_seq_len': 512, 'embed_dim': 512, 'num_heads': 8, 'num_layers': 12, 'd_ff': 2730, 'dropout': 0.1, 'lora_r': 32, 'lora_alpha': 64, 'lora_dropout': 0.05, 'target_modules': ['qkv_proj', 'out_proj'], 'batch_size': 8, 'grad_accum_steps': 4, 'lr': 0.0002, 'warmup_steps': 200, 'epochs': 5, 'grad_clip': 0.8, 'eval_every_steps': 200, 'save_every_steps': 200, 'device': 'cuda'}


In [2]:
# Cell 2 — Tokenizer & dataset

In [3]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

TRAIN_PATH = os.path.join(BASE_DIR, "data", "processed", "train.jsonl")
VAL_PATH = os.path.join(BASE_DIR, "data", "processed", "val_filtered.jsonl")

class JsonlTextDataset(Dataset):
    def __init__(self, path, tokenizer, max_len):
        self.examples = []
        with open(path, encoding="utf-8") as f:
            for line in f:
                text = json.loads(line)["text"] + tokenizer.eos_token
                ids = tokenizer.encode(text, truncation=True, max_length=max_len)
                self.examples.append(ids)

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]

def collate_fn(batch, pad_id, max_len):
    max_batch_len = min(max(len(x) for x in batch), max_len)
    input_ids = torch.full((len(batch), max_batch_len), pad_id, dtype=torch.long)
    labels = torch.full((len(batch), max_batch_len), -100, dtype=torch.long)

    for i, ids in enumerate(batch):
        ids = ids[:max_batch_len]
        input_ids[i, :len(ids)] = torch.tensor(ids)

        # Shift labels: label at position t is the token at position t+1
        if len(ids) > 1:
            labels[i, :len(ids)-1] = torch.tensor(ids[1:])
        # last position has no next-token target, stays -100 (ignored in loss)

    return input_ids, labels

train_dataset = JsonlTextDataset(TRAIN_PATH, tokenizer, CONFIG["max_seq_len"])
val_dataset = JsonlTextDataset(VAL_PATH, tokenizer, CONFIG["max_seq_len"])

print(f"Train examples: {len(train_dataset)}")
print(f"Val examples: {len(val_dataset)}")

Train examples: 45236
Val examples: 924


In [4]:
# Cell 3 — DataLoaders

In [5]:
from functools import partial

collate = partial(collate_fn, pad_id=tokenizer.eos_token_id, max_len=CONFIG["max_seq_len"])

train_loader = DataLoader(
    train_dataset, batch_size=CONFIG["batch_size"], shuffle=True, collate_fn=collate
)
val_loader = DataLoader(
    val_dataset, batch_size=CONFIG["batch_size"], shuffle=False, collate_fn=collate
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

Train batches: 5655
Val batches: 116


In [6]:
# Sanity check: confirm labels are shifted correctly
batch = next(iter(train_loader))
input_ids, labels = batch
print("input_ids[0][:10]:", input_ids[0][:10].tolist())
print("labels[0][:10]:   ", labels[0][:10].tolist())
print("\nExpected: labels[0][:9] should equal input_ids[0][1:10]")
print("Match:", input_ids[0][1:10].tolist() == labels[0][:9].tolist())

input_ids[0][:10]: [21017, 46486, 25, 198, 18438, 391, 262, 3721, 286, 41590]
labels[0][:10]:    [46486, 25, 198, 18438, 391, 262, 3721, 286, 41590, 29163]

Expected: labels[0][:9] should equal input_ids[0][1:10]
Match: True


In [7]:
# Cell 4 — Model architecture (RoPE + SwiGLU transformer)

In [8]:
class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len):
        super().__init__()
        inv_freq = 1.0 / (10000 ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq)

        t = torch.arange(max_seq_len).float()
        freqs = torch.einsum("i,j->ij", t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer("cos_cached", emb.cos()[None, None, :, :])
        self.register_buffer("sin_cached", emb.sin()[None, None, :, :])

    def forward(self, x, seq_len):
        return self.cos_cached[:, :, :seq_len, :], self.sin_cached[:, :, :seq_len, :]

def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)

def apply_rope(q, k, cos, sin):
    q = (q * cos) + (rotate_half(q) * sin)
    k = (k * cos) + (rotate_half(k) * sin)
    return q, k

class SwiGLU(nn.Module):
    def __init__(self, dim, d_ff, dropout):
        super().__init__()
        self.w1 = nn.Linear(dim, d_ff, bias=False)
        self.w2 = nn.Linear(dim, d_ff, bias=False)
        self.w3 = nn.Linear(d_ff, dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.w3(F.silu(self.w1(x)) * self.w2(x)))

class CausalSelfAttention(nn.Module):
    def __init__(self, dim, num_heads, max_seq_len, dropout):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.qkv_proj = nn.Linear(dim, dim * 3, bias=False)
        self.out_proj = nn.Linear(dim, dim, bias=False)
        self.rope = RotaryEmbedding(self.head_dim, max_seq_len)
        self.dropout = dropout

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv_proj(x).reshape(B, T, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        cos, sin = self.rope(x, T)
        q, k = apply_rope(q, k, cos, sin)

        out = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=self.dropout if self.training else 0.0)
        out = out.transpose(1, 2).reshape(B, T, C)
        return self.out_proj(out)

class TransformerBlock(nn.Module):
    def __init__(self, dim, num_heads, d_ff, max_seq_len, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(dim)
        self.attn = CausalSelfAttention(dim, num_heads, max_seq_len, dropout)
        self.ln2 = nn.LayerNorm(dim)
        self.ffn = SwiGLU(dim, d_ff, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

class ENGLlmModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.token_embed = nn.Embedding(cfg["vocab_size"], cfg["embed_dim"])
        self.blocks = nn.ModuleList([
            TransformerBlock(cfg["embed_dim"], cfg["num_heads"], cfg["d_ff"], cfg["max_seq_len"], cfg["dropout"])
            for _ in range(cfg["num_layers"])
        ])
        self.ln_f = nn.LayerNorm(cfg["embed_dim"])
        self.lm_head = nn.Linear(cfg["embed_dim"], cfg["vocab_size"], bias=False)
        # NOTE: not weight-tied — checkpoint has lm_head as a separate parameter

    def forward(self, input_ids):
        x = self.token_embed(input_ids)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        return self.lm_head(x)

print("Model class defined (names matched to checkpoint).")

Model class defined (names matched to checkpoint).


In [9]:
# Cell 5 — LoRA layer wrapper

In [10]:
class LoRALinear(nn.Module):
    def __init__(self, base_linear, r, alpha, dropout):
        super().__init__()
        self.base = base_linear
        self.base.weight.requires_grad = False

        in_dim = base_linear.in_features
        out_dim = base_linear.out_features

        self.lora_A = nn.Parameter(torch.zeros(r, in_dim))
        self.lora_B = nn.Parameter(torch.zeros(out_dim, r))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))

        self.scaling = alpha / r
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        base_out = self.base(x)
        lora_out = self.dropout(x) @ self.lora_A.T @ self.lora_B.T
        return base_out + lora_out * self.scaling

def apply_lora(model, target_modules, r, alpha, dropout):
    for block in model.blocks:
        if "qkv_proj" in target_modules:
            block.attn.qkv_proj = LoRALinear(block.attn.qkv_proj, r, alpha, dropout)
        if "out_proj" in target_modules:
            block.attn.out_proj = LoRALinear(block.attn.out_proj, r, alpha, dropout)
    return model

print("LoRA wrapper defined.")

LoRA wrapper defined.


In [11]:
# Cell 6 — Load base checkpoint & apply LoRA

In [12]:
# Dummy placeholder so pickle can reconstruct the saved config object
class GPTConfig:
    pass

BASE_CHECKPOINT_PATH = r"C:\Users\MIND & MATTER\Desktop\LLM-ENGLISH\Base Model\Best-upgraded_code\base model_training_upgraded_v2\Best Epoch\base_model.pt"

model = ENGLlmModel(CONFIG)

checkpoint = torch.load(BASE_CHECKPOINT_PATH, map_location="cpu", weights_only=False)
state_dict = checkpoint["model_state_dict"]

missing, unexpected = model.load_state_dict(state_dict, strict=False)
print("Missing keys:", missing)
print("Unexpected keys:", unexpected)

# Freeze all base params
for p in model.parameters():
    p.requires_grad = False

model = apply_lora(model, CONFIG["target_modules"], CONFIG["lora_r"], CONFIG["lora_alpha"], CONFIG["lora_dropout"])
model.to(CONFIG["device"])

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} ({100*trainable/total:.2f}% of {total:,})")

Missing keys: []
Unexpected keys: []
Trainable params: 1,179,648 (1.02% of 115,570,688)


In [13]:
# Cell 7 — Optimizer, scheduler, loss

In [14]:
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=CONFIG["lr"], betas=(0.9, 0.95), weight_decay=0.01)

total_steps = (len(train_loader) // CONFIG["grad_accum_steps"]) * CONFIG["epochs"]

def lr_lambda(step):
    if step < CONFIG["warmup_steps"]:
        return step / max(1, CONFIG["warmup_steps"])
    progress = (step - CONFIG["warmup_steps"]) / max(1, total_steps - CONFIG["warmup_steps"])
    return 0.5 * (1 + math.cos(math.pi * min(progress, 1.0)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

def compute_loss(logits, labels):
    return F.cross_entropy(
        logits.view(-1, logits.size(-1)),
        labels.view(-1),
        ignore_index=-100,
    )

print(f"Total optimizer steps planned: {total_steps}")

Total optimizer steps planned: 7065


In [15]:
# Cell 8 — Checkpoint save/load helpers (with resume support)

In [16]:
CKPT_DIR = os.path.join(BASE_DIR, "checkpoints", "stage_a")
LOG_PATH = os.path.join(BASE_DIR, "logs", "stage_a_log.json")

def save_checkpoint(step, epoch, best_val_loss):
    ckpt = {
        "step": step,
        "epoch": epoch,
        "best_val_loss": best_val_loss,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
    }
    path = os.path.join(CKPT_DIR, "latest.pt")
    torch.save(ckpt, path)

    if best_val_loss is not None:
        best_path = os.path.join(CKPT_DIR, "best", "best.pt")
        torch.save(ckpt, best_path)

def load_checkpoint_if_exists():
    path = os.path.join(CKPT_DIR, "latest.pt")
    if not os.path.exists(path):
        print("No checkpoint found — starting fresh.")
        return 0, 0, float("inf")

    ckpt = torch.load(path, map_location=CONFIG["device"])
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])

    print(f"Resumed from step {ckpt['step']}, epoch {ckpt['epoch']}, best_val_loss {ckpt['best_val_loss']:.4f}")
    return ckpt["step"], ckpt["epoch"], ckpt["best_val_loss"]

def log_metrics(entry):
    logs = []
    if os.path.exists(LOG_PATH):
        with open(LOG_PATH) as f:
            logs = json.load(f)
    logs.append(entry)
    with open(LOG_PATH, "w") as f:
        json.dump(logs, f, indent=2)

In [17]:
# Cell 9 — Eval loop

In [18]:
@torch.no_grad()
def evaluate():
    model.eval()
    total_loss, total_batches = 0.0, 0
    for input_ids, labels in val_loader:
        input_ids, labels = input_ids.to(CONFIG["device"]), labels.to(CONFIG["device"])
        logits = model(input_ids)
        loss = compute_loss(logits, labels)
        total_loss += loss.item()
        total_batches += 1
    model.train()
    avg_loss = total_loss / total_batches
    ppl = math.exp(avg_loss)
    return avg_loss, ppl

In [19]:
# Dummy placeholder so pickle can reconstruct the saved config object
class GPTConfig:
    pass

checkpoint = torch.load(
    r"C:\Users\MIND & MATTER\Desktop\LLM-ENGLISH\Base Model\Best-upgraded_code\base model_training_upgraded_v2\Best Epoch\base_model.pt",
    map_location="cpu",
    weights_only=False
)
print(type(checkpoint))
if isinstance(checkpoint, dict) and "config" in checkpoint:
   if isinstance(checkpoint, dict):
    print(list(checkpoint.keys())[:20])


<class 'dict'>
['epoch', 'model_state_dict', 'optimizer_state_dict', 'scheduler_state_dict', 'scaler_state_dict', 'train_loss', 'val_loss', 'train_losses', 'val_losses', 'config']


In [20]:
print(list(state_dict.keys()))

['token_embed.weight', 'blocks.0.ln1.weight', 'blocks.0.ln1.bias', 'blocks.0.ln2.weight', 'blocks.0.ln2.bias', 'blocks.0.attn.qkv_proj.weight', 'blocks.0.attn.out_proj.weight', 'blocks.0.attn.rope.inv_freq', 'blocks.0.attn.rope.cos_cached', 'blocks.0.attn.rope.sin_cached', 'blocks.0.ffn.w1.weight', 'blocks.0.ffn.w2.weight', 'blocks.0.ffn.w3.weight', 'blocks.1.ln1.weight', 'blocks.1.ln1.bias', 'blocks.1.ln2.weight', 'blocks.1.ln2.bias', 'blocks.1.attn.qkv_proj.weight', 'blocks.1.attn.out_proj.weight', 'blocks.1.attn.rope.inv_freq', 'blocks.1.attn.rope.cos_cached', 'blocks.1.attn.rope.sin_cached', 'blocks.1.ffn.w1.weight', 'blocks.1.ffn.w2.weight', 'blocks.1.ffn.w3.weight', 'blocks.2.ln1.weight', 'blocks.2.ln1.bias', 'blocks.2.ln2.weight', 'blocks.2.ln2.bias', 'blocks.2.attn.qkv_proj.weight', 'blocks.2.attn.out_proj.weight', 'blocks.2.attn.rope.inv_freq', 'blocks.2.attn.rope.cos_cached', 'blocks.2.attn.rope.sin_cached', 'blocks.2.ffn.w1.weight', 'blocks.2.ffn.w2.weight', 'blocks.2.ffn.w3

In [21]:
start_step, start_epoch, best_val_loss = load_checkpoint_if_exists()
global_step = start_step
print(f"Will resume training from global_step={global_step}, epoch={start_epoch}")

Resumed from step 4400, epoch 3, best_val_loss 4.4706
Will resume training from global_step=4400, epoch=3


In [21]:
# import os
# for f in ["latest.pt", "best/best.pt"]:
#     p = os.path.join(BASE_DIR, "checkpoints", "stage_a", f)
#     if os.path.exists(p):
#         os.remove(p)
#         print(f"Removed: {p}")
#     else:
#         print(f"Not found (already clean): {p}")

Not found (already clean): C:\Users\MIND & MATTER\Desktop\ScratchLLM-95_conversational\checkpoints\stage_a\latest.pt
Not found (already clean): C:\Users\MIND & MATTER\Desktop\ScratchLLM-95_conversational\checkpoints\stage_a\best/best.pt


In [22]:
import time

model.train()
patience_counter = 0
PATIENCE = 5

SAMPLE_PROMPT = "What is the capital of France?"

@torch.no_grad()
def quick_sample(prompt, max_new_tokens=40, temperature=0.8, top_k=40):
    model.eval()
    input_text = f"### Instruction:\n{prompt}\n\n### Response:\n"
    input_ids = tokenizer.encode(input_text, return_tensors="pt").to(CONFIG["device"])

    for _ in range(max_new_tokens):
        input_ids_cropped = input_ids[:, -CONFIG["max_seq_len"]:]
        logits = model(input_ids_cropped)
        next_token_logits = logits[0, -1, :] / temperature
        top_k_logits, top_k_indices = torch.topk(next_token_logits, top_k)
        probs = F.softmax(top_k_logits, dim=-1)
        next_token = top_k_indices[torch.multinomial(probs, num_samples=1)]
        input_ids = torch.cat([input_ids, next_token.view(1, 1)], dim=1)
        if next_token.item() == tokenizer.eos_token_id:
            break

    output_text = tokenizer.decode(input_ids[0], skip_special_tokens=True)
    model.train()
    return output_text

training_start_time = time.time()
total_steps_planned = total_steps  # from Cell 12

for epoch in range(start_epoch, CONFIG["epochs"]):
    print(f"\n=== Epoch {epoch + 1}/{CONFIG['epochs']} ===")
    optimizer.zero_grad()
    epoch_start = time.time()

    for i, (input_ids, labels) in enumerate(train_loader):
        input_ids, labels = input_ids.to(CONFIG["device"]), labels.to(CONFIG["device"])

        logits = model(input_ids)
        loss = compute_loss(logits, labels) / CONFIG["grad_accum_steps"]
        loss.backward()

        # Lightweight heartbeat every 50 raw batches so it never looks frozen
        if i % 50 == 0:
            elapsed = time.time() - epoch_start
            print(f"  batch {i}/{len(train_loader)} | elapsed {elapsed:.1f}s | loss {loss.item()*CONFIG['grad_accum_steps']:.4f}")

        if (i + 1) % CONFIG["grad_accum_steps"] == 0:
            torch.nn.utils.clip_grad_norm_(trainable_params, CONFIG["grad_clip"])
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1

            if global_step % CONFIG["eval_every_steps"] == 0:
                val_loss, val_ppl = evaluate()
                train_loss_display = loss.item() * CONFIG["grad_accum_steps"]

                # Time tracking / ETA
                elapsed_total = time.time() - training_start_time
                steps_done = global_step - start_step
                avg_time_per_step = elapsed_total / max(1, steps_done)
                steps_remaining = total_steps_planned - global_step
                eta_seconds = steps_remaining * avg_time_per_step
                eta_minutes = eta_seconds / 60

                print(f"Step {global_step}/{total_steps_planned} | train_loss {train_loss_display:.4f} | val_loss {val_loss:.4f} | val_ppl {val_ppl:.2f}")
                print(f"  Elapsed: {elapsed_total/60:.1f} min | Avg/step: {avg_time_per_step:.2f}s | ETA remaining: {eta_minutes:.1f} min")

                # Live sample generation
                sample_output = quick_sample(SAMPLE_PROMPT)
                print(f"  Sample generation:\n  {sample_output}\n")

                log_metrics({
                    "step": global_step,
                    "epoch": epoch + 1,
                    "train_loss": train_loss_display,
                    "val_loss": val_loss,
                    "val_ppl": val_ppl,
                    "lr": scheduler.get_last_lr()[0],
                    "sample_output": sample_output,
                })

                is_best = val_loss < best_val_loss
                if is_best:
                    best_val_loss = val_loss
                    patience_counter = 0
                else:
                    patience_counter += 1

                save_checkpoint(global_step, epoch, best_val_loss if is_best else None)

                if patience_counter >= PATIENCE:
                    print("Val loss not improving — consider stopping manually.")

    save_checkpoint(global_step, epoch + 1, best_val_loss)
    print(f"Epoch {epoch + 1} complete. Checkpoint saved.")

total_time_min = (time.time() - training_start_time) / 60
print(f"\nTraining loop finished (or interrupted — rerun resume + training cells to continue).")
print(f"Total training time this run: {total_time_min:.1f} minutes")


=== Epoch 4/5 ===
  batch 0/5655 | elapsed 0.7s | loss 4.2100
  batch 50/5655 | elapsed 9.4s | loss 4.9861
  batch 100/5655 | elapsed 17.8s | loss 4.8251
  batch 150/5655 | elapsed 26.8s | loss 4.4926
  batch 200/5655 | elapsed 34.8s | loss 4.8913
  batch 250/5655 | elapsed 44.6s | loss 4.7401
  batch 300/5655 | elapsed 52.9s | loss 5.3479
  batch 350/5655 | elapsed 59.8s | loss 4.1952
  batch 400/5655 | elapsed 67.1s | loss 4.5222
  batch 450/5655 | elapsed 77.5s | loss 4.2687
  batch 500/5655 | elapsed 90.0s | loss 4.9641
  batch 550/5655 | elapsed 96.6s | loss 4.2243
  batch 600/5655 | elapsed 103.8s | loss 4.5315
  batch 650/5655 | elapsed 110.9s | loss 5.0586
  batch 700/5655 | elapsed 124.7s | loss 4.1913
  batch 750/5655 | elapsed 131.6s | loss 4.1155
Step 4600/7065 | train_loss 4.6308 | val_loss 4.4672 | val_ppl 87.11
  Elapsed: 2.5 min | Avg/step: 0.76s | ETA remaining: 31.1 min
  Sample generation:
  ### Instruction:
What is the capital of France?

### Response:
The capital 